# Juraj's playground

<img src="image-20260214-150825.png" width="50" align="left" />

<hr>

In [13]:
!pip install polars==1.38.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.4/810.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 66.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [14]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import time
import os
import duckdb
import polars as pl


con = duckdb.connect()
con.execute("SET memory_limit='3GB';")
con.execute("SET threads=2;")
con.execute("SET temp_directory='/work/data/tmp';")

In [10]:
subset = con.execute("""
    SELECT
        COUNT(DISTINCT fs.matas_id) AS sales_ids,
        COUNT(DISTINCT dm.matas_id) AS matched_member_ids
    FROM '/work/data/parquets/fact_sales_v2.parquet' fs
    LEFT JOIN '/work/data/parquets/dim_member_v2.parquet' dm
        ON fs.matas_id = dm.matas_id
""").arrow()

pl.from_arrow(subset)

sales_ids,matched_member_ids
i64,i64
573421,573421


In [11]:
subset = con.execute("""
    SELECT
        COUNT(DISTINCT fs.matas_id) AS total_sales_ids,
        COUNT(DISTINCT dm.matas_id) AS matched_member_ids,
        COUNT(DISTINCT fs.matas_id) 
            - COUNT(DISTINCT dm.matas_id) AS missing_member_ids
    FROM '/work/data/parquets/fact_sales_v2.parquet' fs
    LEFT JOIN '/work/data/parquets/dim_member_v2.parquet' dm
        ON fs.matas_id = dm.matas_id
""").arrow()

pl.from_arrow(subset)

total_sales_ids,matched_member_ids,missing_member_ids
i64,i64,i64
573421,573421,0


In [2]:
input_file = "fact_sales_v2.csv"
output_file = "fact_sales_v2_pandas.parquet"

# Remove file if it already exists
if os.path.exists(output_file):
    os.remove(output_file)

chunksize = 20000
writer = None

start = time.perf_counter()

for chunk in pd.read_csv(input_file, chunksize=chunksize):
    table = pa.Table.from_pandas(chunk)

    if writer is None:
        writer = pq.ParquetWriter(output_file, table.schema)

    writer.write_table(table)

if writer:
    writer.close()

end = time.perf_counter()

print(f"Pandas chunked conversion time: {end - start:.2f} seconds")

Pandas chunked conversion time: 110.81 seconds


In [3]:
input_file = "fact_sales_v2.csv"
output_file = "fact_sales_v2_duckdb.parquet"

if os.path.exists(output_file):
    os.remove(output_file)

con = duckdb.connect()

start = time.perf_counter()

con.execute(f"""
COPY (SELECT * FROM '{input_file}')
TO '{output_file}'
(FORMAT 'parquet');
""")

end = time.perf_counter()

print(f"DuckDB conversion time: {end - start:.2f} seconds")

DuckDB conversion time: 19.71 seconds


In [4]:
import os

print("Pandas parquet size (MB):", os.path.getsize("fact_sales_v2_pandas.parquet") / 1e6)
print("DuckDB parquet size (MB):", os.path.getsize("fact_sales_v2_duckdb.parquet") / 1e6)

Pandas parquet size (MB): 624.973358
DuckDB parquet size (MB): 485.610844


In [ ]:
import pandas as pd

pandas_df = pd.read_parquet("fact_sales_v2_pandas.parquet")
duck_df = pd.read_parquet("fact_sales_v2_duckdb.parquet")

print("Pandas rows:", len(pandas_df))
print("DuckDB rows:", len(duck_df))

<hr>

## Verifying data integrity SQL \(local\) <\> Polars

Validate row counts

In [1]:
import polars as pl

web_sessions = pl.scan_parquet(
    "/work/data/parquets/fact_web_sessions_v2.parquet"
)

row_count_polars = web_sessions.select(
    pl.len().alias("row_count")
).collect()

row_count_polars

row_count
u32
38703196


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_1 = _dntk.execute_sql(
  'SELECT COUNT(*) AS row_count\nFROM fact_web_sessions_v2;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_1

Validate Distinct Counts

In [4]:
distinct_check = (
    web_sessions
    .select([
        pl.col("matas_id").n_unique().alias("unique_matas"),
        pl.col("session_id").n_unique().alias("unique_sessions")
    ])
    .collect()
)

distinct_check

unique_matas,unique_sessions
u32,u32
567329,7221749


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_2 = _dntk.execute_sql(
  'SELECT \n    COUNT(DISTINCT matas_id) AS unique_matas,\n    COUNT(DISTINCT session_id) AS unique_sessions\nFROM fact_web_sessions_v2;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_2

Validate aggregation

In [13]:
polars_result = (
    web_sessions
    .filter(pl.col("event_date") >= pl.date(2024, 1, 1))
    .group_by("matas_id")
    .agg([
        pl.len().alias("session_count"),
        pl.col("session_id").n_unique().alias("unique_sessions")
    ])
    .sort("matas_id")
    .collect()
)

polars_result

matas_id,session_count,unique_sessions
str,u32,u32
"""+++EOcOLigaZxJ2L7AOHGqXdilj0HH…",45,21
"""+++Le73vDenTAPMMjT7YE5HKd9dD9F…",20,11
"""+++Mj6Doy1/v2cpFI4fihaXdilj0HH…",13,11
"""+++Pxtv9xenOfbdNlQXksNlb6+InBv…",4,4
"""+++X5q94AdRqPuRbs0Z+Rdlb6+InBv…",16,16
…,…,…
"""zzwkHNhV4mJBZllAe9LgsJHKd9dD9F…",1,1
"""zzyW2cQvZHcxNajWUnZrZ6Xdilj0HH…",6,5
"""zzz5ehX1+X/BNUBwF5t17aXdilj0HH…",9,9


In [ ]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_3 = _dntk.execute_sql(
  'SELECT\n    matas_id,\n    COUNT(*) AS session_count,\n    COUNT(DISTINCT session_id) AS unique_sessions\nFROM fact_web_sessions_v2\nWHERE event_date >= \'2024-01-01\'\nGROUP BY matas_id\nORDER BY matas_id;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_3

<hr>

In [4]:
subset = con.execute("""
SELECT
    event_name,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN session_id IS NULL THEN 1 ELSE 0 END) AS null_sessions,
    ROUND(
        SUM(CASE WHEN session_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS pct_null_sessions
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
GROUP BY event_name
ORDER BY total_rows DESC
""").arrow()

pl.from_arrow(subset)

event_name,total_rows,null_sessions,pct_null_sessions
str,i64,"decimal[38,0]",f64
"""page_view""",31177047,2763,0.01
"""purchase""",5063116,230191,4.55
"""add_to_cart""",2463033,48260,1.96


In [7]:
subset = con.execute("""
SELECT item_ids
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
WHERE item_ids IS NOT NULL
LIMIT 20
""").arrow()

pl.from_arrow(subset)

item_ids
str
"""728210,728134,739354"""
"""728243,739354"""
"""761287"""
"""640552"""
"""833909,820028,761287"""
…
"""728213,728214"""
"""833909"""
"""833909"""


In [10]:
subset = con.execute("""
WITH session_counts AS (
    SELECT
        session_id,
        COUNT(*) AS event_count
    FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
    WHERE session_id IS NOT NULL
    GROUP BY session_id
)
SELECT
    COUNT(*) AS total_sessions,
    AVG(event_count) AS avg_events,
    MIN(event_count) AS min_events,
    MAX(event_count) AS max_events,
    APPROX_QUANTILE(event_count, 0.5) AS median_events,
    APPROX_QUANTILE(event_count, 0.9) AS p90_events
FROM session_counts
""").arrow()

pl.from_arrow(subset)

total_sessions,avg_events,min_events,max_events,median_events,p90_events
i64,f64,i64,i64,i64,i64
7221748,5.320316,1,901,2,13


In [13]:
subset = con.execute("""
SELECT
    platform,
    event_name,
    COUNT(*) AS event_count
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
GROUP BY platform, event_name
ORDER BY platform, event_count DESC
""").arrow()

pl.from_arrow(subset)

platform,event_name,event_count
str,str,i64
"""ANDROID""","""purchase""",38922
"""ANDROID""","""add_to_cart""",30013
"""IOS""","""purchase""",279753
"""IOS""","""add_to_cart""",135474
"""WEB""","""page_view""",31177047
"""WEB""","""purchase""",4744441
"""WEB""","""add_to_cart""",2297546


In [16]:
subset = con.execute("""
SELECT
    event_name,
    COUNT(*) AS total_rows,
    COUNT(item_ids) AS rows_with_item,
    ROUND(COUNT(item_ids) * 100.0 / COUNT(*), 2) AS pct_with_item
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
GROUP BY event_name
""").arrow()

pl.from_arrow(subset)

event_name,total_rows,rows_with_item,pct_with_item
str,i64,i64,f64
"""add_to_cart""",2463033,2438075,98.99
"""page_view""",31177047,0,0.0
"""purchase""",5063116,5063116,100.0


In [22]:
brand_check = con.execute("""
SELECT
    event_name,
    COUNT(*) AS total_events,
    SUM(CASE WHEN page_view_brand IS NOT NULL THEN 1 ELSE 0 END) AS brand_events
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
GROUP BY event_name
ORDER BY total_events DESC
""").arrow()

pl.from_arrow(brand_check)

event_name,total_events,brand_events
str,i64,"decimal[38,0]"
"""page_view""",31177047,13528781
"""purchase""",5063116,0
"""add_to_cart""",2463033,1587877


In [28]:
brand_check = con.execute("""
SELECT
    page_view_brand,
    COUNT(*) AS views
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
WHERE page_view_brand IS NOT NULL
GROUP BY page_view_brand
ORDER BY views DESC
LIMIT 20
""").arrow()

pl.from_arrow(brand_check)

page_view_brand,views
str,i64
"""Matas Striber""",956554
"""Nilens Jord""",492794
"""e.l.f.""",358424
"""Clinique""",347489
"""DIOR""",331083
…,…
"""Yves Saint Laurent""",170269
"""Clarins""",167264
"""Nailster""",153347


In [37]:
categories = con.execute("""
SELECT
    page_view_category,
    COUNT(*) AS cnt
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
WHERE page_view_category IS NOT NULL
GROUP BY page_view_category
ORDER BY cnt DESC
LIMIT 40
""").arrow()

pl.from_arrow(categories)

page_view_category,cnt
str,i64
"""Eau de Parfum""",639327
"""Dagcreme""",476318
"""Foundation""",351703
"""Serum""",334470
"""Lipgloss""",333561
…,…
"""Deodoranter""",94334
"""Øjenbryn""",89880
"""Natcreme""",89846


In [40]:
rare_categories = con.execute("""
SELECT
    COUNT(*) AS num_categories
FROM (
    SELECT page_view_category, COUNT(*) AS cnt
    FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
    WHERE page_view_category IS NOT NULL
    GROUP BY page_view_category
) t
WHERE cnt <= 10
""").arrow()

pl.from_arrow(rare_categories)

num_categories
i64
17259


In [43]:
rare_examples = con.execute("""
SELECT
    page_view_category,
    COUNT(*) AS cnt
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
WHERE page_view_category IS NOT NULL
GROUP BY page_view_category
HAVING COUNT(*) <= 5
ORDER BY cnt ASC
LIMIT 50
""").arrow()

pl.from_arrow(rare_examples)

page_view_category,cnt
str,i64
"""Alle Tilbud / Alt L'Oréal Pari…",1
"""Alle mærker / Rexona / Hudplej…",1
"""Alle mærker / Malin+Goetz / Mæ…",1
"""Alle mærker / âme pure / Hudpl…",1
"""Alle mærker / Cellu-Cup / Hudp…",1
…,…
"""Alle Tilbud / Ekstra Plus tilb…",1
"""Alle Tilbud / Matas Striber 3 …",1
"""Alle mærker / Biotherm / Blue …",1


In [49]:
date_mismatch = con.execute("""
SELECT COUNT(*) AS mismatches
FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
WHERE CAST(event_timestamp AS DATE) != event_date
""").arrow()

pl.from_arrow(date_mismatch)

mismatches
i64
0


In [17]:
# ==========================================================
# CHECK IF matas_id IN WEB SESSIONS MATCHES MEMBERS
# ==========================================================

# parquet paths
PATH_PARQUETS = "/work/data/parquets"

web_sessions_path = f"{PATH_PARQUETS}/fact_web_sessions_v2.parquet"
members_path = f"{PATH_PARQUETS}/dim_member_v2.parquet"

print("="*60)
print("CHECKING matas_id MATCH BETWEEN WEB SESSIONS AND MEMBERS")
print("="*60)

# ----------------------------------------------------------
# UNIQUE COUNTS
# ----------------------------------------------------------
counts = con.execute(f"""
SELECT
    (SELECT COUNT(DISTINCT matas_id) FROM '{web_sessions_path}') AS unique_web_ids,
    (SELECT COUNT(DISTINCT matas_id) FROM '{members_path}') AS unique_member_ids
""").fetchdf()

print("\nUnique ID counts:")
print(counts)

# ----------------------------------------------------------
# OVERLAP BETWEEN TABLES
# ----------------------------------------------------------
overlap = con.execute(f"""
SELECT
    COUNT(DISTINCT w.matas_id) AS web_ids,
    COUNT(DISTINCT m.matas_id) AS member_ids,
    COUNT(DISTINCT CASE WHEN m.matas_id IS NOT NULL THEN w.matas_id END) AS overlapping_ids
FROM '{web_sessions_path}' w
LEFT JOIN '{members_path}' m
ON w.matas_id = m.matas_id
""").fetchdf()

print("\nID overlap:")
print(overlap)

# ----------------------------------------------------------
# WEB IDS THAT ARE NOT MEMBERS
# ----------------------------------------------------------
not_members = con.execute(f"""
SELECT COUNT(DISTINCT w.matas_id) AS web_ids_not_in_members
FROM '{web_sessions_path}' w
LEFT JOIN '{members_path}' m
ON w.matas_id = m.matas_id
WHERE m.matas_id IS NULL
""").fetchdf()

print("\nWeb IDs not found in members:")
print(not_members)

# ----------------------------------------------------------
# MATCH PERCENTAGE
# ----------------------------------------------------------
web_ids = overlap["web_ids"][0]
overlap_ids = overlap["overlapping_ids"][0]

match_pct = overlap_ids / web_ids * 100 if web_ids else 0

print("\nMatch percentage:")
print(f"{match_pct:.2f}% of web session IDs match member IDs")

# ----------------------------------------------------------
# CHECK ID FORMAT (STRING LENGTH)
# ----------------------------------------------------------
print("\nChecking ID format (string length distribution)")

web_format = con.execute(f"""
SELECT LENGTH(matas_id) AS id_length, COUNT(*) AS count
FROM '{web_sessions_path}'
GROUP BY id_length
ORDER BY count DESC
LIMIT 10
""").fetchdf()

member_format = con.execute(f"""
SELECT LENGTH(matas_id) AS id_length, COUNT(*) AS count
FROM '{members_path}'
GROUP BY id_length
ORDER BY count DESC
LIMIT 10
""").fetchdf()

print("\nWeb session ID format:")
print(web_format)

print("\nMember ID format:")
print(member_format)

print("\nDONE")

CHECKING matas_id MATCH BETWEEN WEB SESSIONS AND MEMBERS

Unique ID counts:
   unique_web_ids  unique_member_ids
0          567329             573421

ID overlap:
   web_ids  member_ids  overlapping_ids
0   567329      567329           567329

Web IDs not found in members:
   web_ids_not_in_members
0                       0

Match percentage:
100.00% of web session IDs match member IDs

Checking ID format (string length distribution)

Web session ID format:
   id_length     count
0         44  37040024
1         24   1663172

Member ID format:
   id_length   count
0         44  556502
1         24   16919

DONE


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=1615b33a-d424-41db-9b01-6976e7db6ad0' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>